In [1]:
import os, gc, warnings

# Ensure protobuf uses pure-Python implementation (helps avoid TF/protobuf ABI issues on Kaggle)
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION_VERSION"] = "2"
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from glob import glob
from tqdm import tqdm

SEED = 42
np.random.seed(SEED)

try:
    import cv2  # noqa: F401

    _HAS_CV2 = True
except Exception:
    _HAS_CV2 = False
    from PIL import Image

USING_TF_KERAS_FALLBACK = False
tf = None
keras = None
K = None
load_model = None

# --- Bugfix: Some Kaggle images have protobuf where MessageFactory lacks GetPrototype,
# which TF expects. Patch it early so TF import doesn't crash before our fallback.
try:
    import google.protobuf  # noqa: F401
    from google.protobuf.message_factory import MessageFactory  # noqa: F401

    if not hasattr(MessageFactory, "GetPrototype") and hasattr(
        MessageFactory, "GetMessageClass"
    ):
        MessageFactory.GetPrototype = MessageFactory.GetMessageClass  # type: ignore[attr-defined]
except Exception:
    pass

try:
    try:
        import tensorflow as tf  # noqa: F401
        from tensorflow import keras  # noqa: F401
        from tensorflow.keras import backend as K  # noqa: F401
        from tensorflow.keras.models import load_model  # noqa: F401
    except AttributeError as e:
        # Keep prior behavior: if TF import still hits protobuf incompatibility, fall back.
        if "GetPrototype" in str(e) or "MessageFactory" in str(e):
            raise ImportError(f"TensorFlow/protobuf incompatibility: {e}") from e
        raise

    try:
        tf.random.set_seed(SEED)
    except Exception:
        pass
    print("TF:", getattr(tf, "__version__", "unknown"))
    USING_TF_KERAS_FALLBACK = False
except Exception as e:
    msg = repr(e)
    print("TensorFlow import failed, falling back to tf_keras. Error:", msg)
    import tf_keras as keras  # type: ignore
    from tf_keras import backend as K  # type: ignore
    from tf_keras.models import load_model  # type: ignore

    tf = None
    USING_TF_KERAS_FALLBACK = True
    print("Using tf_keras fallback:", getattr(keras, "__version__", "unknown"))



2026-05-15 09:20:50.683511: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778836850.695017      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778836850.698220      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

TF: 2.18.0


In [2]:
BATCH_SIZE = 16
EPOCHS = 30
n_splits = 5
fold_selected = 1  # 1..5



In [3]:
df = pd.read_csv(
    "../input/uw-madison-gi-tract-image-segmentation/sample_submission.csv"
)
DEBUG = False
if df.shape[0] == 0:
    DEBUG = True

if DEBUG:
    df = pd.read_csv("../input/uw-madison-gi-tract-image-segmentation/train.csv")
    df = df.drop(columns=["segmentation"])
    df["predicted"] = ""



In [4]:
df = df.rename(columns={"class": "class_name"})
df["case"] = df["id"].apply(lambda x: int(x.split("_")[0].replace("case", "")))
df["day"] = df["id"].apply(lambda x: int(x.split("_")[1].replace("day", "")))
df["slice"] = df["id"].apply(lambda x: x.split("_")[3])

if DEBUG:
    TRAIN_DIR = "../input/uw-madison-gi-tract-image-segmentation/train"
else:
    TRAIN_DIR = "../input/uw-madison-gi-tract-image-segmentation/test"

all_images = glob(os.path.join(TRAIN_DIR, "**", "*.png"), recursive=True)
if len(all_images) == 0:
    raise FileNotFoundError(f"No .png files found under {TRAIN_DIR}")

base = all_images[0].rsplit("/", 4)[0]

df["path_partial"] = [
    os.path.join(
        base,
        f"case{df['case'].values[i]}",
        f"case{df['case'].values[i]}_day{df['day'].values[i]}",
        "scans",
        f"slice_{df['slice'].values[i]}",
    )
    for i in range(df.shape[0])
]

tmp_df = pd.DataFrame(
    {
        "path_partial": [p.rsplit("_", 4)[0] for p in all_images],
        "path": all_images,
    }
)

df = df.merge(tmp_df, on="path_partial", how="left").drop(columns=["path_partial"])
if df["path"].isna().any():
    missing = df[df["path"].isna()].head(5)[["id", "class_name"]]
    raise RuntimeError(f"Failed to resolve some image paths. Examples:\n{missing}")

df["width"] = df["path"].apply(lambda x: int(x[:-4].rsplit("_", 4)[1]))
df["height"] = df["path"].apply(lambda x: int(x[:-4].rsplit("_", 4)[2]))

df.head(3)



,id,class_name,predicted,case,day,slice,path,width,height
0,case123_day20_slice_0001,large_bowel,1 1 5 2,123,20,0001,../input/uw-madison-gi-tract-image-segmentatio...,266,266
1,case123_day20_slice_0001,small_bowel,1 1 5 2,123,20,0001,../input/uw-madison-gi-tract-image-segmentatio...,266,266
2,case123_day20_slice_0001,stomach,1 1 5 2,123,20,0001,../input/uw-madison-gi-tract-image-segmentatio...,266,266


In [5]:
df_train = pd.DataFrame({"id": df["id"][::3].values})
df_train["path"] = df["path"][::3].values
df_train["predicted"] = df["predicted"][::3].values
df_train["case"] = df["case"][::3].values
df_train["day"] = df["day"][::3].values
df_train["slice"] = df["slice"][::3].values
df_train["width"] = df["width"][::3].values
df_train["height"] = df["height"][::3].values

del df
gc.collect()

df_train = df_train.reset_index(drop=True)
df_train = df_train.fillna("")
print(df_train.shape)
df_train.head(3)



(6800, 8)


,id,path,predicted,case,day,slice,width,height
0,case123_day20_slice_0001,../input/uw-madison-gi-tract-image-segmentatio...,1 1 5 2,123,20,0001,266,266
1,case123_day20_slice_0002,../input/uw-madison-gi-tract-image-segmentatio...,1 1 5 2,123,20,0002,266,266
2,case123_day20_slice_0003,../input/uw-madison-gi-tract-image-segmentatio...,1 1 5 2,123,20,0003,266,266


In [6]:
gc.collect()




0

In [7]:
def rle_encode(img):
    """
    img: 2D numpy array (H,W), 1 - mask, 0 - background
    Returns run length as space-delimited string in Kaggle GI Tract format.
    """
    if img is None:
        return ""
    pixels = img.flatten(order="F")  # IMPORTANT: Fortran order for this competition
    pixels = np.concatenate([[0], pixels, [0]]).astype(np.uint8)
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return " ".join(str(x) for x in runs)


def rle_decode(mask_rle, shape, color=1):
    """
    mask_rle: run-length string (start length)
    shape: (height,width,channels) or (height,width)
    Returns: numpy array with exactly the requested shape.
    """
    if isinstance(shape, tuple) and len(shape) == 2:
        h, w = shape
        c = 1
        out_shape = (h, w)
    else:
        h, w, c = shape
        out_shape = (h, w, c)

    if (
        mask_rle is None
        or mask_rle == ""
        or (isinstance(mask_rle, float) and np.isnan(mask_rle))
    ):
        return np.zeros(out_shape, dtype=np.float32)

    s = mask_rle.split()
    starts, lengths = [np.asarray(x, dtype=int) for x in (s[0:][::2], s[1:][::2])]
    starts -= 1
    ends = starts + lengths

    img = np.zeros((h * w, c), dtype=np.float32)
    for lo, hi in zip(starts, ends):
        img[lo:hi, :] = color

    if len(out_shape) == 2:
        return img.reshape((h, w), order="F")
    return img.reshape((h, w, c), order="F")




In [8]:
class DataGenerator(keras.utils.Sequence):
    """
    Minimal bugfixes:
    - ensure X has 3 channels (original code allocated (..,3) but loaded 1 channel)
    - keep behavior: resize to (128,128) and scale to [0,1]
    - allow training dataframe with per-class RLE columns
    """

    def __init__(self, df, batch_size=BATCH_SIZE, subset="train", shuffle=False):
        super().__init__()
        self.df = df.reset_index(drop=True)
        self.shuffle = shuffle
        self.subset = subset
        self.batch_size = batch_size
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))

    def on_epoch_end(self):
        self.indexes = np.arange(len(self.df))
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __getitem__(self, index):
        batch_idx = self.indexes[
            index * self.batch_size : (index + 1) * self.batch_size
        ]
        bs = len(batch_idx)
        X = np.empty((bs, 128, 128, 3), dtype=np.float32)

        if self.subset == "train":
            y = np.empty((bs, 128, 128, 3), dtype=np.float32)

        for i, row_idx in enumerate(batch_idx):
            img_path = self.df.loc[row_idx, "path"]
            img = self.__load_grayscale_3ch(img_path)
            X[i] = img

            if self.subset == "train":
                w = int(self.df.loc[row_idx, "width"])
                h = int(self.df.loc[row_idx, "height"])
                for k, cls in enumerate(["large_bowel", "small_bowel", "stomach"]):
                    rles = self.df.loc[row_idx, cls]
                    masks = rle_decode(rles, shape=(h, w, 1))
                    masks2d = masks[:, :, 0]
                    masks2d = self.__resize_mask(masks2d, (128, 128))
                    y[i, :, :, k] = masks2d

        if self.subset == "train":
            return X, y
        return X

    def __resize_img(self, img, dsize):
        if _HAS_CV2:
            return cv2.resize(img, dsize, interpolation=cv2.INTER_AREA)
        pil = Image.fromarray(img)
        pil = pil.resize(dsize, resample=Image.BILINEAR)
        return np.array(pil)

    def __resize_mask(self, mask, dsize_hw):
        H, W = dsize_hw
        if _HAS_CV2:
            return cv2.resize(mask, (W, H), interpolation=cv2.INTER_NEAREST)
        pil = Image.fromarray((mask * 255).astype(np.uint8))
        pil = pil.resize((W, H), resample=Image.NEAREST)
        return (np.array(pil) > 127).astype(np.float32)

    def __load_grayscale_3ch(self, img_path):
        if _HAS_CV2:
            img = cv2.imread(img_path, cv2.IMREAD_ANYDEPTH)
            if img is None:
                raise FileNotFoundError(f"Failed to read image: {img_path}")
        else:
            with Image.open(img_path) as im:
                img = np.array(im)
        img = self.__resize_img(img, (128, 128))
        img = img.astype(np.float32)
        mx = img.max() if img.max() > 0 else 1.0
        img = img / mx
        img = np.expand_dims(img, axis=-1)  # (H,W,1)
        img = np.repeat(img, 3, axis=-1)  # (H,W,3)
        return img




In [9]:
def dice_coef(y_true, y_pred, smooth=1):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2.0 * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)


def iou_coef(y_true, y_pred, smooth=1):
    intersection = K.sum(K.abs(y_true * y_pred), axis=[1, 2, 3])
    union = K.sum(y_true, [1, 2, 3]) + K.sum(y_pred, [1, 2, 3]) - intersection
    iou = K.mean((intersection + smooth) / (union + smooth), axis=0)
    return iou


def dice_loss(y_true, y_pred):
    smooth = 1.0
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = y_true_f * y_pred_f
    score = (2.0 * K.sum(intersection) + smooth) / (
        K.sum(y_true_f) + K.sum(y_pred_f) + smooth
    )
    return 1.0 - score


def bce_dice_loss(y_true, y_pred):
    y_true = K.cast(y_true, "float32")
    return keras.losses.binary_crossentropy(y_true, y_pred) + 0.5 * dice_loss(
        y_true, y_pred
    )


class FixedDropout(keras.layers.Dropout):
    def _get_noise_shape(self, inputs):
        if self.noise_shape is None:
            return self.noise_shape
        symbolic_shape = K.shape(inputs)
        noise_shape = [
            symbolic_shape[axis] if shape is None else shape
            for axis, shape in enumerate(self.noise_shape)
        ]
        return tuple(noise_shape)




In [10]:
custom_objects = {
    "FixedDropout": FixedDropout,
    "dice_coef": dice_coef,
    "iou_coef": iou_coef,
    "bce_dice_loss": bce_dice_loss,
}

model_path = "../input/uwmgi-unet-keras/model.h5"

model = None
if os.path.exists(model_path):
    model = load_model(model_path, custom_objects=custom_objects)
    print("Loaded external model:", model_path)
else:
    print("External model not found; will train a small fallback model from train.csv.")

gc.collect()



External model not found; will train a small fallback model from train.csv.


0

In [11]:
if model is None:
    train_csv = pd.read_csv("../input/uw-madison-gi-tract-image-segmentation/train.csv")
    train_piv = (
        train_csv.pivot_table(
            index="id", columns="class", values="segmentation", aggfunc="first"
        )
        .reset_index()
        .rename_axis(None, axis=1)
    )
    tdf = train_piv.copy()
    tdf["case"] = tdf["id"].apply(lambda x: int(x.split("_")[0].replace("case", "")))
    tdf["day"] = tdf["id"].apply(lambda x: int(x.split("_")[1].replace("day", "")))
    tdf["slice"] = tdf["id"].apply(lambda x: x.split("_")[3])

    TRAIN_DIR_TR = "../input/uw-madison-gi-tract-image-segmentation/train"
    all_tr_images = glob(os.path.join(TRAIN_DIR_TR, "**", "*.png"), recursive=True)
    if len(all_tr_images) == 0:
        raise FileNotFoundError(f"No .png files found under {TRAIN_DIR_TR}")

    base_tr = all_tr_images[0].rsplit("/", 4)[0]
    tdf["path_partial"] = [
        os.path.join(
            base_tr,
            f"case{tdf['case'].values[i]}",
            f"case{tdf['case'].values[i]}_day{tdf['day'].values[i]}",
            "scans",
            f"slice_{tdf['slice'].values[i]}",
        )
        for i in range(tdf.shape[0])
    ]
    tmp_tr = pd.DataFrame(
        {
            "path_partial": [p.rsplit("_", 4)[0] for p in all_tr_images],
            "path": all_tr_images,
        }
    )
    tdf = tdf.merge(tmp_tr, on="path_partial", how="left").drop(
        columns=["path_partial"]
    )
    tdf = tdf.dropna(subset=["path"]).reset_index(drop=True)

    tdf["width"] = tdf["path"].apply(lambda x: int(x[:-4].rsplit("_", 4)[1]))
    tdf["height"] = tdf["path"].apply(lambda x: int(x[:-4].rsplit("_", 4)[2]))
    for c in ["large_bowel", "small_bowel", "stomach"]:
        if c not in tdf.columns:
            tdf[c] = ""

    tdf = tdf.sample(n=min(2500, len(tdf)), random_state=SEED).reset_index(drop=True)

    val_frac = 0.1
    n_val = max(1, int(len(tdf) * val_frac))
    val_df = tdf.iloc[:n_val].reset_index(drop=True)
    tr_df = tdf.iloc[n_val:].reset_index(drop=True)

    train_gen = DataGenerator(
        tr_df, batch_size=BATCH_SIZE, subset="train", shuffle=True
    )
    val_gen = DataGenerator(
        val_df, batch_size=BATCH_SIZE, subset="train", shuffle=False
    )

    def conv_block(x, f):
        x = keras.layers.Conv2D(f, 3, padding="same")(x)
        x = keras.layers.BatchNormalization()(x)
        x = keras.layers.Activation("relu")(x)
        x = keras.layers.Conv2D(f, 3, padding="same")(x)
        x = keras.layers.BatchNormalization()(x)
        x = keras.layers.Activation("relu")(x)
        return x

    def build_unet(input_shape=(128, 128, 3), out_ch=3):
        inp = keras.Input(shape=input_shape)
        c1 = conv_block(inp, 16)
        p1 = keras.layers.MaxPooling2D()(c1)

        c2 = conv_block(p1, 32)
        p2 = keras.layers.MaxPooling2D()(c2)

        c3 = conv_block(p2, 64)
        p3 = keras.layers.MaxPooling2D()(c3)

        bn = conv_block(p3, 128)

        u3 = keras.layers.UpSampling2D()(bn)
        u3 = keras.layers.Concatenate()([u3, c3])
        c4 = conv_block(u3, 64)

        u2 = keras.layers.UpSampling2D()(c4)
        u2 = keras.layers.Concatenate()([u2, c2])
        c5 = conv_block(u2, 32)

        u1 = keras.layers.UpSampling2D()(c5)
        u1 = keras.layers.Concatenate()([u1, c1])
        c6 = conv_block(u1, 16)

        out = keras.layers.Conv2D(out_ch, 1, activation="sigmoid")(c6)
        return keras.Model(inp, out)

    model = build_unet()
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3), loss=bce_dice_loss, metrics=[dice_coef]
    )

    epochs_to_run = min(EPOCHS, 3)
    model.fit(train_gen, validation_data=val_gen, epochs=epochs_to_run, verbose=1)

    del (
        train_csv,
        train_piv,
        tdf,
        tr_df,
        val_df,
        train_gen,
        val_gen,
        all_tr_images,
        tmp_tr,
    )
    gc.collect()



2026-05-15 09:20:58.961341: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Epoch 1/3


2026-05-15 09:21:07.005041: E tensorflow/core/util/util.cc:131] oneDNN supports DT_INT32 only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


141/141 ━━━━━━━━━━━━━━━━━━━━ 55s 330ms/step - dice_coef: 0.0278 - loss: 0.9149 - val_dice_coef: 0.0246 - val_loss: 0.7432
Epoch 2/3
141/141 ━━━━━━━━━━━━━━━━━━━━ 46s 324ms/step - dice_coef: 0.0543 - loss: 0.6310 - val_dice_coef: 0.0608 - val_loss: 0.5939
Epoch 3/3
141/141 ━━━━━━━━━━━━━━━━━━━━ 46s 325ms/step - dice_coef: 0.1079 - loss: 0.5330 - val_dice_coef: 0.1279 - val_loss: 0.5295


In [12]:
pred_batches = DataGenerator(df_train, batch_size=1, subset="test", shuffle=False)
gc.collect()
LOGITS = model.predict(pred_batches, verbose=1)
gc.collect()
print("Pred shape:", LOGITS.shape)



6800/6800 ━━━━━━━━━━━━━━━━━━━━ 54s 8ms/step
Pred shape: (6800, 128, 128, 3)


In [13]:
PRED_THRESH = 0.50

lbs, sbs, sts = [], [], []

n_pred = LOGITS.shape[0]
if n_pred != len(df_train):
    min_len = min(n_pred, len(df_train))
    LOGITS = LOGITS[:min_len]
    df_train = df_train.iloc[:min_len].reset_index(drop=True)

for index in tqdm(range(len(df_train)), total=len(df_train)):
    h = int(df_train.loc[index, "height"])
    w = int(df_train.loc[index, "width"])

    if _HAS_CV2:
        pred0 = cv2.resize(
            LOGITS[index, :, :, 0], (w, h), interpolation=cv2.INTER_NEAREST
        )
        pred1 = cv2.resize(
            LOGITS[index, :, :, 1], (w, h), interpolation=cv2.INTER_NEAREST
        )
        pred2 = cv2.resize(
            LOGITS[index, :, :, 2], (w, h), interpolation=cv2.INTER_NEAREST
        )
    else:
        pred0 = (
            np.array(
                Image.fromarray((LOGITS[index, :, :, 0] * 255).astype(np.uint8)).resize(
                    (w, h), resample=Image.NEAREST
                ),
                dtype=np.uint8,
            ).astype(np.float32)
            / 255.0
        )
        pred1 = (
            np.array(
                Image.fromarray((LOGITS[index, :, :, 1] * 255).astype(np.uint8)).resize(
                    (w, h), resample=Image.NEAREST
                ),
                dtype=np.uint8,
            ).astype(np.float32)
            / 255.0
        )
        pred2 = (
            np.array(
                Image.fromarray((LOGITS[index, :, :, 2] * 255).astype(np.uint8)).resize(
                    (w, h), resample=Image.NEAREST
                ),
                dtype=np.uint8,
            ).astype(np.float32)
            / 255.0
        )

    pred0 = (pred0 > PRED_THRESH).astype("uint8")
    pred1 = (pred1 > PRED_THRESH).astype("uint8")
    pred2 = (pred2 > PRED_THRESH).astype("uint8")

    lbs.append(rle_encode(pred0))
    sbs.append(rle_encode(pred1))
    sts.append(rle_encode(pred2))

del LOGITS
gc.collect()

print(len(lbs), len(sbs), len(sts), "images encoded")



100%|██████████| 6800/6800 [00:04<00:00, 1645.15it/s]

6800 6800 6800 images encoded


In [14]:
ids, classes, rles = [], [], []
for index in tqdm(range(len(df_train)), total=len(df_train)):
    img_id = df_train.loc[index, "id"]
    ids.extend([img_id] * 3)
    classes.extend(["large_bowel", "small_bowel", "stomach"])
    rles.extend([lbs[index], sbs[index], sts[index]])

submission = pd.DataFrame({"id": ids, "class": classes, "predicted": rles})

sample = pd.read_csv(
    "../input/uw-madison-gi-tract-image-segmentation/sample_submission.csv"
)

submission = sample[["id", "class"]].merge(submission, on=["id", "class"], how="left")
submission["predicted"] = submission["predicted"].fillna("")

submission.to_csv("submission.csv", index=False)
print(submission.shape)
submission.head()



100%|██████████| 6800/6800 [00:00<00:00, 233281.81it/s]

(20400, 3)


,id,class,predicted
0,case123_day20_slice_0001,large_bowel,25596 2 25600 2 25862 2 25866 2 26126 12 26392...
1,case123_day20_slice_0001,small_bowel,18035 9 18301 9 18567 11 18833 11 19099 17 193...
2,case123_day20_slice_0001,stomach,
3,case123_day20_slice_0002,large_bowel,26660 8 26926 8 27190 12 27456 12 27720 16 279...
4,case123_day20_slice_0002,small_bowel,21153 4 21419 4 21685 4 21951 4 22217 4 22481 ...


In [15]:
sample = pd.read_csv(
    "../input/uw-madison-gi-tract-image-segmentation/sample_submission.csv"
)
print("sample:", sample.shape, "submission:", submission.shape)
print("columns:", submission.columns.tolist())
print("saved to:", os.path.abspath("submission.csv"))
print("missing predicted:", submission["predicted"].isna().sum())
print("empty predicted:", (submission["predicted"] == "").sum())
print("Using tf_keras fallback:", USING_TF_KERAS_FALLBACK)
print("Has cv2:", _HAS_CV2)
print("Prediction threshold:", PRED_THRESH)

sample: (20400, 3) submission: (20400, 3)
columns: ['id', 'class', 'predicted']
saved to: /kaggle/working/submission.csv
missing predicted: 0
empty predicted: 8601
Using tf_keras fallback: False
Has cv2: True
Prediction threshold: 0.5
